# CinePal — TMDB Snapshot & GPU Embedding on Colab

Runs the full `tmdb_snapshot → split → embed → upload to HF` pipeline on a
free Colab GPU. Each run publishes a timestamped set of parquet files
(`main_YYYYMMDD.parquet`, `mini_YYYYMMDD.parquet`, `eval_holdout_YYYYMMDD.parquet`).
Paste the printed filenames into `configs/default.yaml` under `ingestion.artifacts`
to pin the next experimental condition to this snapshot.

Local ingest then loads them with no further changes:

```bash
python -m db.ingest             # mini
python -m db.ingest --set main  # full set
```

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Click the 🔑 *Secrets* icon (left sidebar) and add:
   - `TMDB_API_KEY` — from https://www.themoviedb.org/settings/api (v3 auth)
   - `HF_TOKEN` — huggingface.co → Settings → Access Tokens (write scope)
   - `GITHUB_TOKEN` *(only if the repo is private)* — a PAT with `repo` scope
3. Fill in `REPO_URL` in cell 3.

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# 1 — Verify GPU is available
!nvidia-smi

In [ ]:
# 2 — Install project dependencies
!pip install -q sentence-transformers pyarrow pandas numpy huggingface_hub httpx pyyaml python-dotenv tqdm psycopg[binary] pgvector annotated-types pydantic pydantic-settings

In [ ]:
# 3 — Clone the repo and add it to the Python path
import os
import sys
from google.colab import userdata

REPO_URL = "https://github.com/YOUR_ORG/cantucci.git"  # <-- fill in before running

# For private repos, inject the GitHub token into the clone URL.
try:
    github_token = userdata.get("GITHUB_TOKEN")
    if github_token:
        REPO_URL = REPO_URL.replace("https://", f"https://{github_token}@")
except Exception:
    pass  # secret not set — assume public repo

if not os.path.isdir("cantucci"):
    os.system(f"git clone {REPO_URL} cantucci")
else:
    os.system("git -C cantucci pull --ff-only")

os.chdir("cantucci")
if "." not in sys.path:
    sys.path.insert(0, ".")

print("Working directory:", os.getcwd())

In [ ]:
# 4 — Surface the required secrets as env vars so backend.settings can read them
for key in ("TMDB_API_KEY", "HF_TOKEN"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception as exc:
        raise RuntimeError(f"Missing Colab secret: {key}") from exc
print("Secrets loaded.")

In [ ]:
# 5 — Load project settings from configs/default.yaml
from backend.settings import get_settings, ARTIFACTS_DIR

cfg = get_settings()
print(
    f"Config loaded — embedding model: {cfg.representation.model}, "
    f"dim: {cfg.representation.embedding_dim}, split seed: {cfg.split.seed}, "
    f"mini_size: {cfg.split.mini_size}"
)

In [ ]:
# 6 — Snapshot TMDB → cleaned DataFrame
from db.ingestion.tmdb_snapshot import snapshot
from db.ingestion import split

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# For a smoke run before committing to a full pull (~hour), set limit=1000.
df = snapshot(min_vote_count=5, limit=None)
print(f"Snapshot rows: {len(df)}")

main_df, mini_df, eval_df = split.three_way(
    df,
    mini_size=cfg.split.mini_size,
    eval_frac=cfg.split.eval_frac,
    seed=cfg.split.seed,
)
print(f"Splits — main: {len(main_df)}, mini: {len(mini_df)}, eval: {len(eval_df)}")

In [ ]:
# 7 — Embed on GPU
import gc
import torch
from sentence_transformers import SentenceTransformer

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")

model = SentenceTransformer(cfg.representation.model, device=device)
model.half()  # cast weights to float16 — halves VRAM, works universally

def encode_split(split_df, name):
    texts = list(split_df["composite_text"])
    print(f"Encoding {name}: {len(texts)} texts...")
    with torch.no_grad():
        emb = model.encode(
            texts,
            batch_size=64,               # safe for T4
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")
    gc.collect()
    torch.cuda.empty_cache()
    return emb

main_emb = encode_split(main_df, "main")
mini_emb = encode_split(mini_df, "mini")
eval_emb = encode_split(eval_df, "eval")

for name, split_df, emb in [("main", main_df, main_emb), ("mini", mini_df, mini_emb), ("eval", eval_df, eval_emb)]:
    assert emb.shape == (len(split_df), cfg.representation.embedding_dim), f"{name}: unexpected shape {emb.shape}"
    print(f"{name}: {emb.shape}")

In [ ]:
# 8 — Upload timestamped artifacts to Hugging Face
from db.ingestion.upload import upload_artifacts

files = upload_artifacts(
    main_df, mini_df, eval_df,
    main_emb, mini_emb, eval_emb,
    repo_id=cfg.ingestion.hf_repo,
    artifacts_dir=ARTIFACTS_DIR,
)

print("\nPublished. Paste these into configs/default.yaml under `ingestion.artifacts`:\n")
for split_name, filename in files.items():
    print(f"  {split_name}: {filename}")

## Done

1. Copy the filenames printed above into `configs/default.yaml`:

   ```yaml
   ingestion:
     hf_repo: "446f6e6e79/CinePal-embeddings"
     artifacts:
       main: main_YYYYMMDD.parquet
       mini: mini_YYYYMMDD.parquet
       eval_holdout: eval_holdout_YYYYMMDD.parquet
   ```

2. Teammates ingest without a GPU:

   ```bash
   python -m db.ingest             # mini
   python -m db.ingest --set main  # full set
   ```

> **Note on GPU vs CPU floating-point:** embedding outputs may differ by ~1e-6 between
> devices. This is below any meaningful threshold for cosine similarity and can be ignored.